<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/notebooks/01-foundations/2026-10-02-Normalize-and-Join-Nested-JSON-Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (one item inside one order)

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(orders_json, record_path='lines',
                          meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


I wanted one row per line item so I pointed record_path at lines, then used meta to keep order and vendor_id on every row so I still know where each item came from. Got 4 rows and the qty adds up to 7 like it should.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched.groupby('vendor_id')['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
vendor_id
V-07    3
Name: qty, dtype: int64


V-07 is the vendor that did not match, and it has 3 units attached to it. I used a left join to keep every line item even when the vendor isn't in the vendor list, and merge shows V-07 as left_only, meaning it's in the orders but not in vendors. If I did an inner join those 3 would've disappeared.


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# V-07 has no name, and groupby drops missing keys, so it would disappear
# from the table. I'm labeling it UNKNOWN (V-07) so it still shows and the total
# stays 7.
joined['vendor_name'] = joined['name'].fillna('UNKNOWN (' + joined['vendor_id'] + ')')
by_vendor = joined.groupby('vendor_name')['qty'].agg(total_qty='sum')
print(by_vendor)

                total_qty
vendor_name              
Cav Merch               1
Hoos Burgers            3
UNKNOWN (V-07)          3


V-07 doesn't have a name, and groupby throws out missing keys, so it would've dropped out of the table. I labeled it UNKNOWN (V-07) so it still shows up and everything still adds to 7.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
# dropna = False keeps the no-zone row visible instead of dropping it
by_zone = joined.groupby('zone', dropna=False)['qty'].agg(total_qty='sum')
print(by_zone)

zone_total = joined.loc[joined['zone'].notna(), 'qty'].sum()
print('zone total:', zone_total, 'of', joined['qty'].sum())

top_zone = by_zone[by_zone.index.notna()]['total_qty'].idxmax()
print('top zone:', top_zone)

      total_qty
zone           
A             4
NaN           3
zone total: 4 of 7
top zone: A


Zone A sold the most with 4 items. My zone total is 4 though, not 7. The missing 3 are V-07's hot dogs, since V-07 isn't in the vendor list and so it has no zone. I used dropna=False so that row shows up instead of getting dropped.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},
]

# TODO
rows = []
for order in ragged_json:
    for line in order.get('lines', []):
        rows.append({'order': order['order'],
                     'vendor_id': order['vendor_id'],
                     'item': line['item'],
                     'qty': line.get('qty')})

ragged = pd.DataFrame(rows)
ragged['qty'] = ragged['qty'].astype('Int64')
print(ragged)

print('missing qty:', ragged['qty'].isna().sum())
print('zero qty:', (ragged['qty'] == 0).sum())
print('orders with no lines:', [o['order'] for o in ragged_json if 'lines' not in o])

   order vendor_id   item   qty
0      4      V-01   Soda     2
1      6      V-01  Fries  <NA>
missing qty: 1
zero qty: 0
orders with no lines: [5]


json_normalize crashes on this one because order 5 has no lines key, so I looped through it. .get('lines', []) skips an order with no lines and .get('qty') leaves a missing qty as missing. I made qty an Int64 so Fries shows NA and not 0. Ended up with 1 missing qty, 0 zero qtys, and order 5 has no lines.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape because quantity belongs to an item, not to an order. An order can have more than one item in it, so if I kept one row per order, the items and their quantities would be stuck inside a list in a single cell and pandas couldn't add them up. Flattening puts every quantity in its own row in one qty column. The question that becomes easy is "how many total units did each vendor (or zone) sell." With one row per line item it's a single groupby and sum on qty. With one row per order I'd have to dig into each order's list and add up the quantities by hand before I could group anything.

b) A quantity of zero is a real value: we know the count and it's 0. A missing quantity means the number was never recorded, so we don't know what it is. In Q5 the Fries line was ordered, but we don't know how many. If I treat missing as zero, the total looks complete when it's actually an undercount, because the Fries line gets counted as 0 units when it was an unknown number. Averages get pulled down too, since a fake 0 is now included in the calculation. Additionally, once it's a 0 there's no easy way for me to tell it apart from a real zero, so I can't count how many quantities were missing or go back and fix them. Keeping it as NA means the sum skips it and I can still see that one value is missing.